# Parte 4 — Modelo prescriptivo de reposición

## AndinaRetail S.A.C.

Este notebook implementa un modelo prescriptivo de reposición para enero de 2026.

El modelo utiliza:

- la demanda pronosticada por tienda y categoría;
- el stock final de diciembre de 2025;
- costos de adquisición, almacenamiento y demanda no atendida;
- restricciones de presupuesto, capacidad y nivel de servicio;
- PuLP y el solver CBC.

Se ejecuta un escenario base y ocho escenarios de sensibilidad. Finalmente, se exportan los tres archivos oficiales requeridos para Power BI.

In [2]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
import pulp
import yaml

warnings.filterwarnings("ignore")

SEMILLA = 2026
np.random.seed(SEMILLA)

# Detectar la raíz del repositorio.
RUTA_ACTUAL = Path.cwd()

if (RUTA_ACTUAL / "datos").exists():
    RAIZ = RUTA_ACTUAL
elif (RUTA_ACTUAL.parent / "datos").exists():
    RAIZ = RUTA_ACTUAL.parent
else:
    raise FileNotFoundError(
        "No se encontró la raíz del repositorio. "
        "Ejecuta el notebook desde la raíz o desde la carpeta notebooks."
    )

RUTA_DATOS = RAIZ / "datos"
RUTA_RESULTADOS = RAIZ / "resultados"
RUTA_CONFIG = RAIZ / "config" / "escenarios.yaml"

RUTA_RESULTADOS.mkdir(parents=True, exist_ok=True)

print("Raíz del proyecto:", RAIZ.resolve())
print("Python:", sys.version.split()[0])
print("PuLP:", pulp.__version__)
print("Solver CBC disponible:", bool(pulp.PULP_CBC_CMD().available()))
print("Semilla:", SEMILLA)

Raíz del proyecto: C:\Users\DEYSI\OneDrive - Universidad Nacional Mayor de San Marcos\Documentos\proyecto-andinaretail
Python: 3.13.5
PuLP: 3.3.2
Solver CBC disponible: True
Semilla: 2026


In [3]:
# Cargar configuración YAML
with open(RUTA_CONFIG, "r", encoding="utf-8") as archivo:
    config = yaml.safe_load(archivo)

config_prescriptiva = config["modelado_prescriptivo"]

# Cargar archivos de entrada
predicciones = pd.read_csv(RUTA_RESULTADOS / "predicciones_demanda.csv")
inventario = pd.read_csv(RUTA_DATOS / "inventario.csv")
productos = pd.read_csv(RUTA_DATOS / "productos.csv")
ventas = pd.read_csv(RUTA_DATOS / "ventas.csv")
tiendas = pd.read_csv(RUTA_DATOS / "tiendas.csv")

print("Archivos cargados correctamente")
print("Predicciones:", predicciones.shape)
print("Inventario:", inventario.shape)
print("Productos:", productos.shape)
print("Ventas:", ventas.shape)
print("Tiendas:", tiendas.shape)

print("\nPeriodo de optimización:", config_prescriptiva["periodo_optimizacion"])
print("Solver configurado:", config_prescriptiva["motor_solver"])
print("Escenarios configurados:", len(config_prescriptiva["escenarios"]))

Archivos cargados correctamente
Predicciones: (90, 14)
Inventario: (432000, 8)
Productos: (800, 8)
Ventas: (250000, 12)
Tiendas: (15, 8)

Periodo de optimización: 2026-01
Solver configurado: CBC
Escenarios configurados: 9


In [4]:
# ============================
# Validación de predicciones
# ============================

print("=== VALIDACIÓN DEL INPUT PREDICTIVO ===")

# Periodos presentes
print("\nPeriodos:", predicciones["periodo_objetivo"].unique())

# Duplicados
duplicados = predicciones.duplicated(
    subset=["id_tienda", "categoria", "periodo_objetivo"]
).sum()
print("Duplicados:", duplicados)

# Valores negativos
negativos = (
    (predicciones["demanda_baja"] < 0)
    | (predicciones["demanda_predicha"] < 0)
    | (predicciones["demanda_central"] < 0)
    | (predicciones["demanda_alta"] < 0)
).sum()
print("Registros con demanda negativa:", negativos)

# Orden correcto
orden = (
    (predicciones["demanda_baja"] <= predicciones["demanda_predicha"])
    & (predicciones["demanda_predicha"] <= predicciones["demanda_alta"])
).all()

print("Orden baja ≤ predicha ≤ alta:", orden)

print("\nNúmero de tiendas:", predicciones["id_tienda"].nunique())
print("Número de categorías:", predicciones["categoria"].nunique())

=== VALIDACIÓN DEL INPUT PREDICTIVO ===

Periodos: ['2026-01']
Duplicados: 0
Registros con demanda negativa: 0
Orden baja ≤ predicha ≤ alta: True

Número de tiendas: 15
Número de categorías: 6


In [6]:
# ==========================================
# Stock inicial para la optimización
# ==========================================

# Asegurar que el periodo tenga formato de texto
inventario["periodo"] = inventario["periodo"].astype(str)

# Filtrar inventario de diciembre de 2025
inventario_dic_2025 = inventario[
    inventario["periodo"] == "2025-12"
].copy()

# Incorporar la categoría de cada producto
inventario_dic_2025 = inventario_dic_2025.merge(
    productos[["id_producto", "categoria"]],
    on="id_producto",
    how="left",
    validate="many_to_one"
)

# Agregar stock final por tienda y categoría
stock_inicial = (
    inventario_dic_2025
    .groupby(["id_tienda", "categoria"], as_index=False)
    .agg(stock_inicial=("stock_final", "sum"))
)

print("=== STOCK INICIAL DICIEMBRE 2025 ===")
print("Filas:", len(stock_inicial))
print("Tiendas:", stock_inicial["id_tienda"].nunique())
print("Categorías:", stock_inicial["categoria"].nunique())
print("Stock inicial total:", stock_inicial["stock_inicial"].sum())
print("Categorías faltantes:", stock_inicial["categoria"].isna().sum())

stock_inicial.head()

=== STOCK INICIAL DICIEMBRE 2025 ===
Filas: 90
Tiendas: 15
Categorías: 6
Stock inicial total: 67882
Categorías faltantes: 0


,id_tienda,categoria,stock_inicial
0,APP,Abarrotes,1712
1,APP,Bebidas,1337
2,APP,Cuidado Personal,936
3,APP,Electrohogar,892
4,APP,Hogar,659


In [17]:
# ==================================================
# Costos oficiales por tienda y categoría
# ==================================================

ventas_costos = ventas.copy()
ventas_costos["fecha"] = pd.to_datetime(
    ventas_costos["fecha"],
    errors="coerce"
)

# Ventas del año 2025
ventas_costos = ventas_costos[
    ventas_costos["fecha"].dt.year == 2025
].copy()

# Incorporar categoría y costo unitario
ventas_costos = ventas_costos.merge(
    productos[
        [
            "id_producto",
            "categoria",
            "costo_unitario",
            "precio_lista"
        ]
    ],
    on="id_producto",
    how="left",
    validate="many_to_one"
)

ventas_costos = ventas_costos[
    ventas_costos["cantidad"] > 0
].copy()

# Numeradores ponderados
ventas_costos["costo_adquisicion_ponderado"] = (
    ventas_costos["costo_unitario"]
    * ventas_costos["cantidad"]
)

# --------------------------------------------------
# 1. Costos por tienda y categoría
# --------------------------------------------------

costos_tienda_categoria = (
    ventas_costos
    .groupby(
        ["id_tienda", "categoria"],
        as_index=False
    )
    .agg(
        unidades_vendidas=("cantidad", "sum"),
        costo_adquisicion_total=(
            "costo_adquisicion_ponderado",
            "sum"
        ),
        ingreso_neto_total=("monto_total", "sum")
    )
)

costos_tienda_categoria["costo_adquisicion"] = (
    costos_tienda_categoria["costo_adquisicion_total"]
    / costos_tienda_categoria["unidades_vendidas"]
)

costos_tienda_categoria["costo_faltante"] = (
    costos_tienda_categoria["ingreso_neto_total"]
    / costos_tienda_categoria["unidades_vendidas"]
)

# --------------------------------------------------
# 2. Fallback general por categoría
# --------------------------------------------------

fallback_categoria = (
    ventas_costos
    .groupby("categoria", as_index=False)
    .agg(
        unidades_categoria=("cantidad", "sum"),
        costo_adquisicion_categoria_total=(
            "costo_adquisicion_ponderado",
            "sum"
        ),
        ingreso_categoria_total=("monto_total", "sum")
    )
)

fallback_categoria["costo_adquisicion_fallback"] = (
    fallback_categoria[
        "costo_adquisicion_categoria_total"
    ]
    / fallback_categoria["unidades_categoria"]
)

fallback_categoria["costo_faltante_fallback"] = (
    fallback_categoria["ingreso_categoria_total"]
    / fallback_categoria["unidades_categoria"]
)

fallback_categoria = fallback_categoria[
    [
        "categoria",
        "costo_adquisicion_fallback",
        "costo_faltante_fallback"
    ]
]

# --------------------------------------------------
# 3. Todas las combinaciones tienda-categoría
# --------------------------------------------------

combinaciones = predicciones[
    ["id_tienda", "categoria"]
].drop_duplicates()

costos_modelo = (
    combinaciones
    .merge(
        costos_tienda_categoria[
            [
                "id_tienda",
                "categoria",
                "costo_adquisicion",
                "costo_faltante"
            ]
        ],
        on=["id_tienda", "categoria"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        fallback_categoria,
        on="categoria",
        how="left",
        validate="many_to_one"
    )
)

# Aplicar fallback cuando falte información
costos_modelo["costo_adquisicion"] = (
    costos_modelo["costo_adquisicion"]
    .fillna(
        costos_modelo["costo_adquisicion_fallback"]
    )
)

costos_modelo["costo_faltante"] = (
    costos_modelo["costo_faltante"]
    .fillna(
        costos_modelo["costo_faltante_fallback"]
    )
)

# --------------------------------------------------
# 4. Costo de almacenamiento según tienda
# --------------------------------------------------

costos_modelo = costos_modelo.merge(
    tiendas[
        [
            "id_tienda",
            "tipo",
            "ciudad"
        ]
    ],
    on="id_tienda",
    how="left",
    validate="many_to_one"
)

tasa_base = config_prescriptiva[
    "costos"
]["almacenamiento"]["tasa_base_mensual"]

incremento_trujillo = config_prescriptiva[
    "costos"
]["almacenamiento"]["incremento_trujillo_pct"]

es_trujillo_fisica = (
    (costos_modelo["tipo"] == "Fisica")
    & (costos_modelo["ciudad"] == "Trujillo")
)

costos_modelo["tasa_almacenamiento"] = np.where(
    es_trujillo_fisica,
    tasa_base * (1 + incremento_trujillo),
    tasa_base
)

costos_modelo["costo_almacenamiento"] = (
    costos_modelo["costo_adquisicion"]
    * costos_modelo["tasa_almacenamiento"]
)

costos_modelo = costos_modelo[
    [
        "id_tienda",
        "categoria",
        "costo_adquisicion",
        "costo_faltante",
        "tasa_almacenamiento",
        "costo_almacenamiento"
    ]
]

print("=== COSTOS OFICIALES POR TIENDA Y CATEGORÍA ===")
print("Filas:", len(costos_modelo))
print(
    "Valores faltantes:",
    costos_modelo.isna().sum().sum()
)
print(
    "Tasas de almacenamiento:",
    sorted(
        costos_modelo[
            "tasa_almacenamiento"
        ].unique()
    )
)

costos_modelo.head()

=== COSTOS OFICIALES POR TIENDA Y CATEGORÍA ===
Filas: 90
Valores faltantes: 0
Tasas de almacenamiento: [np.float64(0.02), np.float64(0.026000000000000002)]


,id_tienda,categoria,costo_adquisicion,costo_faltante,tasa_almacenamiento,costo_almacenamiento
0,APP,Abarrotes,34.032184,39.208493,0.02,0.680644
1,APP,Bebidas,25.527220,30.980617,0.02,0.510544
2,APP,Cuidado Personal,70.589908,102.236107,0.02,1.411798
3,APP,Electrohogar,852.785781,1117.257920,0.02,17.055716
4,APP,Hogar,186.085057,259.070544,0.02,3.721701


In [19]:
# ==========================================
# Construcción de la tabla maestra
# ==========================================

modelo = (
    predicciones
    .merge(
        stock_inicial,
        on=["id_tienda", "categoria"],
        how="left",
        validate="one_to_one"
    )
    .merge(
        costos_modelo,
        on=["id_tienda", "categoria"],
        how="left",
        validate="one_to_one"
    )
)

# Completar posibles faltantes
modelo["stock_inicial"] = modelo["stock_inicial"].fillna(0)

print("=== TABLA MAESTRA ACTUALIZADA ===")
print("Filas:", len(modelo))
print("Columnas:", len(modelo.columns))
print("Valores faltantes:", modelo.isna().sum().sum())
print()

print(
    modelo[
        [
            "periodo_objetivo",
            "id_tienda",
            "categoria",
            "demanda_baja",
            "demanda_central",
            "demanda_alta",
            "stock_inicial",
            "costo_adquisicion",
            "costo_faltante",
            "tasa_almacenamiento",
            "costo_almacenamiento"
        ]
    ].head()
)

=== TABLA MAESTRA ACTUALIZADA ===
Filas: 90
Columnas: 19
Valores faltantes: 0

  periodo_objetivo id_tienda         categoria  demanda_baja  demanda_central  \
0          2026-01       APP         Abarrotes           883              919   
1          2026-01       APP           Bebidas           670              706   
2          2026-01       APP  Cuidado Personal           433              470   
3          2026-01       APP      Electrohogar           403              439   
4          2026-01       APP             Hogar           326              362   

   demanda_alta  stock_inicial  costo_adquisicion  costo_faltante  \
0           961           1712          34.032184       39.208493   
1           749           1337          25.527220       30.980617   
2           512            936          70.589908      102.236107   
3           481            892         852.785781     1117.257920   
4           404            659         186.085057      259.070544   

   tasa_almacenamie

In [21]:
# ==========================================
# Parámetros base desde config/escenarios.yaml
# ==========================================

PRESUPUESTO_REFERENCIA = (
    config_prescriptiva["presupuesto"]["factor_base"]
)

FACTOR_HOLGURA_CAPACIDAD = (
    config_prescriptiva["capacidad"]["factor_holgura"]
)

NIVEL_SERVICIO_BASE = (
    config_prescriptiva["nivel_servicio"]["objetivo_base"]
)

FACTORES_ESPACIO = (
    config_prescriptiva["capacidad"]["factores_espacio"]
)

print("=== PARÁMETROS CONFIGURADOS ===")
print("Factor base de presupuesto:", PRESUPUESTO_REFERENCIA)
print("Factor de holgura de capacidad:", FACTOR_HOLGURA_CAPACIDAD)
print("Nivel de servicio base:", NIVEL_SERVICIO_BASE)
print("Factores de espacio:", FACTORES_ESPACIO)
print("Registros del modelo:", len(modelo))

=== PARÁMETROS CONFIGURADOS ===
Factor base de presupuesto: 1.0
Factor de holgura de capacidad: 1.1
Nivel de servicio base: 0.95
Factores de espacio: {'Abarrotes': 1.0, 'Bebidas': 1.2, 'Limpieza': 1.3, 'Cuidado Personal': 0.8, 'Electrohogar': 6.0, 'Hogar': 3.0}
Registros del modelo: 90


In [22]:
# ==========================================
# Revisión de configuración prescriptiva
# ==========================================

print("=== CLAVES DEL MODELO PRESCRIPTIVO ===")
print(config_prescriptiva.keys())

print("\n=== COSTOS ===")
print(config_prescriptiva.get("costos"))

print("\n=== PRESUPUESTO ===")
print(config_prescriptiva.get("presupuesto"))

print("\n=== CAPACIDAD ===")
print(config_prescriptiva.get("capacidad"))

print("\n=== NIVEL DE SERVICIO ===")
print(config_prescriptiva.get("nivel_servicio"))

=== CLAVES DEL MODELO PRESCRIPTIVO ===
dict_keys(['semilla', 'libreria', 'motor_solver', 'periodo_optimizacion', 'granularidad', 'desagregar_a_producto', 'demanda', 'inventario_inicial', 'costos', 'presupuesto', 'capacidad', 'nivel_servicio', 'funcion_objetivo', 'restricciones', 'politica_referencia', 'escenarios', 'configuracion_solver'])

=== COSTOS ===
{'adquisicion': {'metodo': 'promedio_ponderado_por_unidades_vendidas_2025', 'peso': 'ventas.cantidad', 'fallback_1': 'promedio_categoria_negocio', 'fallback_2': 'promedio_simple_catalogo_categoria'}, 'faltante': {'metodo': 'ingreso_neto_unitario_ponderado_2025', 'formula': 'SUM(monto_total) / SUM(cantidad)', 'fallback_1': 'promedio_categoria_negocio', 'fallback_2': 'promedio_simple_categoria'}, 'almacenamiento': {'tasa_base_mensual': 0.02, 'aplicar_incremento_trujillo': True, 'incremento_trujillo_pct': 0.3, 'condicion_trujillo': {'tipo': 'Fisica', 'ciudad': 'Trujillo'}}}

=== PRESUPUESTO ===
{'referencia': 'necesidad_neta_central', 'f

In [23]:
# ==================================================
# Presupuesto base y capacidad equivalente por nodo
# ==================================================

# Factores de espacio configurados por categoría
factores_espacio = config_prescriptiva["capacidad"]["factores_espacio"]
factor_holgura = config_prescriptiva["capacidad"]["factor_holgura"]
factor_presupuesto = config_prescriptiva["presupuesto"]["factor_base"]

# --------------------------------------------------
# 1. Necesidad neta central y presupuesto de referencia
# --------------------------------------------------

modelo["necesidad_neta_central"] = np.maximum(
    modelo["demanda_central"] - modelo["stock_inicial"],
    0
)

modelo["costo_necesidad_neta"] = (
    modelo["necesidad_neta_central"]
    * modelo["costo_adquisicion"]
)

presupuesto_referencia = modelo["costo_necesidad_neta"].sum()
presupuesto_base = presupuesto_referencia * factor_presupuesto

# --------------------------------------------------
# 2. Capacidad histórica equivalente por nodo
# --------------------------------------------------

inventario_capacidad = inventario.copy()

inventario_capacidad = inventario_capacidad.merge(
    productos[["id_producto", "categoria"]],
    on="id_producto",
    how="left",
    validate="many_to_one"
)

inventario_capacidad["factor_espacio"] = (
    inventario_capacidad["categoria"].map(factores_espacio)
)

inventario_capacidad["stock_inicial_equivalente"] = (
    inventario_capacidad["stock_inicial"]
    * inventario_capacidad["factor_espacio"]
)

inventario_capacidad["stock_final_equivalente"] = (
    inventario_capacidad["stock_final"]
    * inventario_capacidad["factor_espacio"]
)

# Agregar por nodo y periodo
capacidad_historica = (
    inventario_capacidad
    .groupby(["id_tienda", "periodo"], as_index=False)
    .agg(
        stock_inicial_equivalente=(
            "stock_inicial_equivalente",
            "sum"
        ),
        stock_final_equivalente=(
            "stock_final_equivalente",
            "sum"
        )
    )
)

# Considerar el máximo entre stock inicial y final
capacidad_historica["stock_equivalente_periodo"] = (
    capacidad_historica[
        [
            "stock_inicial_equivalente",
            "stock_final_equivalente"
        ]
    ].max(axis=1)
)

capacidad_nodo = (
    capacidad_historica
    .groupby("id_tienda", as_index=False)
    .agg(
        maximo_historico_stock_equivalente=(
            "stock_equivalente_periodo",
            "max"
        )
    )
)

capacidad_nodo["capacidad_base"] = (
    capacidad_nodo["maximo_historico_stock_equivalente"]
    * factor_holgura
)

# --------------------------------------------------
# 3. Validaciones
# --------------------------------------------------

print("=== PRESUPUESTO Y CAPACIDAD BASE ===")
print("Demanda central total:", modelo["demanda_central"].sum())
print("Stock inicial total:", modelo["stock_inicial"].sum())
print(
    "Necesidad neta central:",
    modelo["necesidad_neta_central"].sum()
)
print("Presupuesto de referencia:", round(presupuesto_referencia, 2))
print("Presupuesto base:", round(presupuesto_base, 2))
print("Nodos con capacidad:", len(capacidad_nodo))
print("Capacidades faltantes:", capacidad_nodo["capacidad_base"].isna().sum())

capacidad_nodo.head()

=== PRESUPUESTO Y CAPACIDAD BASE ===
Demanda central total: 14619
Stock inicial total: 67882
Necesidad neta central: 0
Presupuesto de referencia: 0.0
Presupuesto base: 0.0
Nodos con capacidad: 15
Capacidades faltantes: 0


,id_tienda,maximo_historico_stock_equivalente,capacidad_base
0,APP,15814.8,17396.28
1,T001,12183.1,13401.41
2,T002,12343.9,13578.29
3,T003,12529.0,13781.90
4,T004,12372.0,13609.20


In [24]:
# ==================================================
# Diagnóstico previo de cobertura y capacidad
# ==================================================

# Factor de espacio por categoría
modelo["factor_espacio"] = (
    modelo["categoria"]
    .map(factores_espacio)
)

# Incorporar capacidad base de cada nodo
modelo = modelo.merge(
    capacidad_nodo[
        [
            "id_tienda",
            "maximo_historico_stock_equivalente",
            "capacidad_base"
        ]
    ],
    on="id_tienda",
    how="left",
    validate="many_to_one"
)

# Necesidad neta por escenario de demanda
modelo["necesidad_neta_baja"] = np.maximum(
    modelo["demanda_baja"] - modelo["stock_inicial"],
    0
)

modelo["necesidad_neta_central"] = np.maximum(
    modelo["demanda_central"] - modelo["stock_inicial"],
    0
)

modelo["necesidad_neta_alta"] = np.maximum(
    modelo["demanda_alta"] - modelo["stock_inicial"],
    0
)

# Stock equivalente por categoría
modelo["stock_inicial_equivalente"] = (
    modelo["stock_inicial"]
    * modelo["factor_espacio"]
)

# Resumen inicial por nodo
diagnostico_nodo = (
    modelo
    .groupby("id_tienda", as_index=False)
    .agg(
        demanda_central=(
            "demanda_central",
            "sum"
        ),
        demanda_alta=(
            "demanda_alta",
            "sum"
        ),
        stock_inicial=(
            "stock_inicial",
            "sum"
        ),
        stock_inicial_equivalente=(
            "stock_inicial_equivalente",
            "sum"
        ),
        capacidad_base=(
            "capacidad_base",
            "first"
        ),
        necesidad_neta_central=(
            "necesidad_neta_central",
            "sum"
        ),
        necesidad_neta_alta=(
            "necesidad_neta_alta",
            "sum"
        )
    )
)

diagnostico_nodo["cobertura_central"] = (
    diagnostico_nodo["stock_inicial"]
    / diagnostico_nodo["demanda_central"]
)

diagnostico_nodo["utilizacion_capacidad_inicial_pct"] = (
    diagnostico_nodo["stock_inicial_equivalente"]
    / diagnostico_nodo["capacidad_base"]
    * 100
)

print("=== DIAGNÓSTICO PREVIO ===")
print(
    "Necesidad neta baja:",
    modelo["necesidad_neta_baja"].sum()
)
print(
    "Necesidad neta central:",
    modelo["necesidad_neta_central"].sum()
)
print(
    "Necesidad neta alta:",
    modelo["necesidad_neta_alta"].sum()
)
print(
    "Nodos que superan capacidad inicial:",
    (
        diagnostico_nodo[
            "stock_inicial_equivalente"
        ]
        > diagnostico_nodo["capacidad_base"]
    ).sum()
)
print(
    "Cobertura central mínima:",
    round(
        diagnostico_nodo["cobertura_central"].min(),
        4
    )
)

diagnostico_nodo.head()

=== DIAGNÓSTICO PREVIO ===
Necesidad neta baja: 0
Necesidad neta central: 0
Necesidad neta alta: 0
Nodos que superan capacidad inicial: 0
Cobertura central mínima: 1.9305


,id_tienda,demanda_central,demanda_alta,stock_inicial,stock_inicial_equivalente,capacidad_base,necesidad_neta_central,necesidad_neta_alta,cobertura_central,utilizacion_capacidad_inicial_pct
0,APP,3353,3606,6473,12612.3,17396.28,0,0,1.930510,72.499983
1,T001,638,890,4600,9040.3,13401.41,0,0,7.210031,67.457827
2,T002,672,925,4601,9099.7,13578.29,0,0,6.846726,67.016539
3,T003,765,1017,4699,9412.1,13781.90,0,0,6.142484,68.293196
4,T004,752,1005,4609,9288.6,13609.20,0,0,6.128989,68.252359


In [25]:
# ==================================================
# Revisión de escenarios y configuración del solver
# ==================================================

print("=== ESCENARIOS CONFIGURADOS ===")

for escenario in config_prescriptiva["escenarios"]:
    print(escenario)

print("\n=== CONFIGURACIÓN DEL SOLVER ===")
print(config_prescriptiva["configuracion_solver"])

print("\n=== RESTRICCIONES CONFIGURADAS ===")
print(config_prescriptiva["restricciones"])

print("\n=== FUNCIÓN OBJETIVO ===")
print(config_prescriptiva["funcion_objetivo"])

=== ESCENARIOS CONFIGURADOS ===
{'id': 'base', 'demanda': 'central', 'factor_presupuesto': 1.0, 'factor_capacidad': 1.0, 'nivel_servicio': 0.95}
{'id': 'demanda_baja', 'demanda': 'baja', 'factor_presupuesto': 1.0, 'factor_capacidad': 1.0, 'nivel_servicio': 0.95}
{'id': 'demanda_alta', 'demanda': 'alta', 'factor_presupuesto': 1.0, 'factor_capacidad': 1.0, 'nivel_servicio': 0.95}
{'id': 'presupuesto_menos_10', 'demanda': 'central', 'factor_presupuesto': 0.9, 'factor_capacidad': 1.0, 'nivel_servicio': 0.95}
{'id': 'presupuesto_mas_10', 'demanda': 'central', 'factor_presupuesto': 1.1, 'factor_capacidad': 1.0, 'nivel_servicio': 0.95}
{'id': 'capacidad_menos_10', 'demanda': 'central', 'factor_presupuesto': 1.0, 'factor_capacidad': 0.9, 'nivel_servicio': 0.95}
{'id': 'capacidad_mas_10', 'demanda': 'central', 'factor_presupuesto': 1.0, 'factor_capacidad': 1.1, 'nivel_servicio': 0.95}
{'id': 'servicio_90', 'demanda': 'central', 'factor_presupuesto': 1.0, 'factor_capacidad': 1.0, 'nivel_servicio

In [27]:
# ==================================================
# Formulación y resolución de un escenario con PuLP
# ==================================================

def resolver_escenario(
    datos_modelo: pd.DataFrame,
    escenario: dict,
    presupuesto_base: float,
    solver_msg: bool = False
):
    """
    Formula y resuelve un escenario del modelo prescriptivo.

    Variables:
    q = unidades a reponer
    u = demanda no atendida
    e = inventario final estimado
    """

    datos = datos_modelo.copy().reset_index(drop=True)

    # ----------------------------------------------
    # 1. Seleccionar demanda del escenario
    # ----------------------------------------------

    columnas_demanda = {
        "baja": "demanda_baja",
        "central": "demanda_central",
        "alta": "demanda_alta"
    }

    tipo_demanda = escenario["demanda"]
    columna_demanda = columnas_demanda[tipo_demanda]

    datos["demanda_escenario"] = (
        datos[columna_demanda]
        .round()
        .clip(lower=0)
        .astype(int)
    )

    # Parámetros del escenario
    factor_presupuesto_escenario = escenario["factor_presupuesto"]
    factor_capacidad_escenario = escenario["factor_capacidad"]
    nivel_servicio_objetivo = escenario["nivel_servicio"]

    presupuesto_escenario = (
        presupuesto_base
        * factor_presupuesto_escenario
    )

    # ----------------------------------------------
    # 2. Crear problema de optimización
    # ----------------------------------------------

    problema = pulp.LpProblem(
        f"reposicion_{escenario['id']}",
        pulp.LpMinimize
    )

    indices = list(datos.index)

    # ----------------------------------------------
    # 3. Variables enteras no negativas
    # ----------------------------------------------

    q = pulp.LpVariable.dicts(
        "reposicion",
        indices,
        lowBound=0,
        cat=pulp.LpInteger
    )

    u = pulp.LpVariable.dicts(
        "demanda_no_atendida",
        indices,
        lowBound=0,
        cat=pulp.LpInteger
    )

    e = pulp.LpVariable.dicts(
        "stock_final_estimado",
        indices,
        lowBound=0,
        cat=pulp.LpInteger
    )

    # ----------------------------------------------
    # 4. Función objetivo
    # ----------------------------------------------

    problema += pulp.lpSum(
        datos.loc[i, "costo_adquisicion"] * q[i]
        + datos.loc[i, "costo_almacenamiento"] * e[i]
        + datos.loc[i, "costo_faltante"] * u[i]
        for i in indices
    )

    # ----------------------------------------------
    # 5. Balance de inventario
    # ----------------------------------------------

    for i in indices:
        problema += (
            datos.loc[i, "stock_inicial"]
            + q[i]
            - datos.loc[i, "demanda_escenario"]
            + u[i]
            == e[i]
        )

        # La demanda no atendida no puede superar la demanda
        problema += (
            u[i] <= datos.loc[i, "demanda_escenario"]
        )

    # ----------------------------------------------
    # 6. Restricción presupuestaria global
    # ----------------------------------------------

    problema += (
        pulp.lpSum(
            datos.loc[i, "costo_adquisicion"] * q[i]
            for i in indices
        )
        <= presupuesto_escenario
    )

    # ----------------------------------------------
    # 7. Capacidad y servicio por nodo
    # ----------------------------------------------

    for id_tienda in datos["id_tienda"].unique():

        indices_tienda = datos.index[
            datos["id_tienda"] == id_tienda
        ].tolist()

        capacidad_base_tienda = datos.loc[
            indices_tienda[0],
            "capacidad_base"
        ]

        capacidad_escenario = (
            capacidad_base_tienda
            * factor_capacidad_escenario
        )

        # Capacidad inmediatamente después de la reposición
        problema += (
            pulp.lpSum(
                datos.loc[i, "factor_espacio"]
                * (
                    datos.loc[i, "stock_inicial"]
                    + q[i]
                )
                for i in indices_tienda
            )
            <= capacidad_escenario
        )

        demanda_total_tienda = datos.loc[
            indices_tienda,
            "demanda_escenario"
        ].sum()

        # Fill rate mínimo por nodo
        problema += (
            pulp.lpSum(
                datos.loc[i, "demanda_escenario"] - u[i]
                for i in indices_tienda
            )
            >= nivel_servicio_objetivo
            * demanda_total_tienda
        )

    # ----------------------------------------------
    # 8. Resolver con CBC
    # ----------------------------------------------

    solver = pulp.PULP_CBC_CMD(msg=solver_msg)
    problema.solve(solver)

    estado_solver = pulp.LpStatus[problema.status]

    # ----------------------------------------------
    # 9. Recuperar solución
    # ----------------------------------------------

    if estado_solver == "Optimal":
        datos["reposicion_recomendada"] = [
            int(round(pulp.value(q[i])))
            for i in indices
        ]

        datos["demanda_no_atendida"] = [
            int(round(pulp.value(u[i])))
            for i in indices
        ]

        datos["stock_final_estimado"] = [
            int(round(pulp.value(e[i])))
            for i in indices
        ]
    else:
        datos["reposicion_recomendada"] = np.nan
        datos["demanda_no_atendida"] = np.nan
        datos["stock_final_estimado"] = np.nan

    datos["demanda_atendida"] = (
        datos["demanda_escenario"]
        - datos["demanda_no_atendida"]
    )

    datos["costo_adquisicion_total"] = (
        datos["reposicion_recomendada"]
        * datos["costo_adquisicion"]
    )

    datos["costo_almacenamiento_total"] = (
        datos["stock_final_estimado"]
        * datos["costo_almacenamiento"]
    )

    datos["costo_faltante_total"] = (
        datos["demanda_no_atendida"]
        * datos["costo_faltante"]
    )

    datos["costo_total"] = (
        datos["costo_adquisicion_total"]
        + datos["costo_almacenamiento_total"]
        + datos["costo_faltante_total"]
    )

    datos["escenario"] = escenario["id"]
    datos["tipo_demanda"] = tipo_demanda
    datos["factor_presupuesto"] = factor_presupuesto_escenario
    datos["factor_capacidad"] = factor_capacidad_escenario
    datos["nivel_servicio_objetivo"] = nivel_servicio_objetivo
    datos["presupuesto_disponible"] = presupuesto_escenario
    datos["estado_solver"] = estado_solver

    return datos

In [28]:
# ==================================================
# Ejecución del escenario base
# ==================================================

escenario_base = next(
    escenario
    for escenario in config_prescriptiva["escenarios"]
    if escenario["id"] == "base"
)

resultado_base = resolver_escenario(
    datos_modelo=modelo,
    escenario=escenario_base,
    presupuesto_base=presupuesto_base,
    solver_msg=False
)

print("=== RESULTADO ESCENARIO BASE ===")
print("Estado del solver:", resultado_base["estado_solver"].iloc[0])
print(
    "Reposición total:",
    resultado_base["reposicion_recomendada"].sum()
)
print(
    "Demanda no atendida total:",
    resultado_base["demanda_no_atendida"].sum()
)
print(
    "Stock final estimado total:",
    resultado_base["stock_final_estimado"].sum()
)
print(
    "Costo total:",
    round(resultado_base["costo_total"].sum(), 2)
)

=== RESULTADO ESCENARIO BASE ===
Estado del solver: Optimal
Reposición total: 0
Demanda no atendida total: 0
Stock final estimado total: 53263
Costo total: 194917.94


In [29]:
# ==================================================
# Validaciones del escenario base
# ==================================================

validacion_base = resultado_base.copy()

# Balance esperado:
# stock inicial + reposición - demanda + faltante = stock final
validacion_base["balance_calculado"] = (
    validacion_base["stock_inicial"]
    + validacion_base["reposicion_recomendada"]
    - validacion_base["demanda_escenario"]
    + validacion_base["demanda_no_atendida"]
)

errores_balance = (
    validacion_base["balance_calculado"]
    != validacion_base["stock_final_estimado"]
).sum()

variables_negativas = (
    (
        validacion_base[
            [
                "reposicion_recomendada",
                "demanda_no_atendida",
                "stock_final_estimado"
            ]
        ] < 0
    )
    .any(axis=1)
    .sum()
)

variables_no_enteras = (
    validacion_base[
        [
            "reposicion_recomendada",
            "demanda_no_atendida",
            "stock_final_estimado"
        ]
    ]
    .apply(lambda columna: (columna % 1 != 0).sum())
    .sum()
)

# Nivel de servicio por nodo
servicio_base_nodo = (
    validacion_base
    .groupby("id_tienda", as_index=False)
    .agg(
        demanda_total=("demanda_escenario", "sum"),
        demanda_atendida=("demanda_atendida", "sum")
    )
)

servicio_base_nodo["nivel_servicio"] = (
    servicio_base_nodo["demanda_atendida"]
    / servicio_base_nodo["demanda_total"]
)

# Uso de capacidad por nodo
capacidad_base_nodo = (
    validacion_base
    .assign(
        inventario_reposicion_equivalente=lambda df:
            (
                df["stock_inicial"]
                + df["reposicion_recomendada"]
            )
            * df["factor_espacio"]
    )
    .groupby("id_tienda", as_index=False)
    .agg(
        inventario_reposicion_equivalente=(
            "inventario_reposicion_equivalente",
            "sum"
        ),
        capacidad_disponible=("capacidad_base", "first")
    )
)

excesos_capacidad = (
    capacidad_base_nodo["inventario_reposicion_equivalente"]
    > capacidad_base_nodo["capacidad_disponible"] + 1e-6
).sum()

presupuesto_utilizado = (
    validacion_base["costo_adquisicion_total"].sum()
)

print("=== VALIDACIONES ESCENARIO BASE ===")
print("Errores de balance:", errores_balance)
print("Variables negativas:", variables_negativas)
print("Variables no enteras:", variables_no_enteras)
print(
    "Nivel de servicio mínimo:",
    round(servicio_base_nodo["nivel_servicio"].min(), 4)
)
print(
    "Nodos bajo objetivo:",
    (
        servicio_base_nodo["nivel_servicio"]
        < escenario_base["nivel_servicio"] - 1e-9
    ).sum()
)
print("Excesos de capacidad:", excesos_capacidad)
print("Presupuesto utilizado:", round(presupuesto_utilizado, 2))
print(
    "Presupuesto disponible:",
    round(resultado_base["presupuesto_disponible"].iloc[0], 2)
)

=== VALIDACIONES ESCENARIO BASE ===
Errores de balance: 0
Variables negativas: 0
Variables no enteras: 0
Nivel de servicio mínimo: 1.0
Nodos bajo objetivo: 0
Excesos de capacidad: 0
Presupuesto utilizado: 0.0
Presupuesto disponible: 0.0


In [30]:
# ==================================================
# Ejecución de los 9 escenarios
# ==================================================

resultados_escenarios = []

for escenario in config_prescriptiva["escenarios"]:
    resultado = resolver_escenario(
        datos_modelo=modelo,
        escenario=escenario,
        presupuesto_base=presupuesto_base,
        solver_msg=False
    )

    resultados_escenarios.append(resultado)

    print(
        escenario["id"],
        "->",
        resultado["estado_solver"].iloc[0],
        "| reposición:",
        resultado["reposicion_recomendada"].sum(),
        "| faltante:",
        resultado["demanda_no_atendida"].sum()
    )

print("\nEscenarios ejecutados:", len(resultados_escenarios))

base -> Optimal | reposición: 0 | faltante: 0
demanda_baja -> Optimal | reposición: 0 | faltante: 0
demanda_alta -> Optimal | reposición: 0 | faltante: 0
presupuesto_menos_10 -> Optimal | reposición: 0 | faltante: 0
presupuesto_mas_10 -> Optimal | reposición: 0 | faltante: 0
capacidad_menos_10 -> Optimal | reposición: 0 | faltante: 0
capacidad_mas_10 -> Optimal | reposición: 0 | faltante: 0
servicio_90 -> Optimal | reposición: 0 | faltante: 0
servicio_98 -> Optimal | reposición: 0 | faltante: 0

Escenarios ejecutados: 9


In [31]:
# ==================================================
# Resumen de escenarios de optimización
# ==================================================

filas_resumen = []

for resultado in resultados_escenarios:
    escenario_id = resultado["escenario"].iloc[0]
    estado_solver = resultado["estado_solver"].iloc[0]

    demanda_total = resultado["demanda_escenario"].sum()
    reposicion_total = resultado["reposicion_recomendada"].sum()
    demanda_atendida_total = resultado["demanda_atendida"].sum()
    faltante_total = resultado["demanda_no_atendida"].sum()

    nivel_servicio_alcanzado = (
        demanda_atendida_total / demanda_total
        if demanda_total > 0
        else 1.0
    )

    presupuesto_disponible = (
        resultado["presupuesto_disponible"].iloc[0]
    )

    presupuesto_utilizado = (
        resultado["costo_adquisicion_total"].sum()
    )

    costo_adquisicion = (
        resultado["costo_adquisicion_total"].sum()
    )

    costo_almacenamiento = (
        resultado["costo_almacenamiento_total"].sum()
    )

    costo_faltante = (
        resultado["costo_faltante_total"].sum()
    )

    costo_total = resultado["costo_total"].sum()

    filas_resumen.append(
        {
            "escenario": escenario_id,
            "tipo_demanda": resultado["tipo_demanda"].iloc[0],
            "factor_presupuesto": resultado[
                "factor_presupuesto"
            ].iloc[0],
            "factor_capacidad": resultado[
                "factor_capacidad"
            ].iloc[0],
            "nivel_servicio_objetivo": resultado[
                "nivel_servicio_objetivo"
            ].iloc[0],
            "estado_solver": estado_solver,
            "demanda_total": demanda_total,
            "reposicion_total": reposicion_total,
            "demanda_atendida": demanda_atendida_total,
            "faltante_total": faltante_total,
            "nivel_servicio_alcanzado_pct": (
                nivel_servicio_alcanzado * 100
            ),
            "presupuesto_disponible": presupuesto_disponible,
            "presupuesto_utilizado": presupuesto_utilizado,
            "costo_adquisicion": costo_adquisicion,
            "costo_almacenamiento": costo_almacenamiento,
            "costo_faltante": costo_faltante,
            "costo_total": costo_total
        }
    )

resumen_escenarios = pd.DataFrame(filas_resumen)

print("=== RESUMEN DE ESCENARIOS ===")
print("Filas:", len(resumen_escenarios))
print(
    "Estados:",
    resumen_escenarios["estado_solver"].unique()
)

resumen_escenarios

=== RESUMEN DE ESCENARIOS ===
Filas: 9
Estados: ['Optimal']


,escenario,tipo_demanda,factor_presupuesto,factor_capacidad,nivel_servicio_objetivo,estado_solver,demanda_total,reposicion_total,demanda_atendida,faltante_total,nivel_servicio_alcanzado_pct,presupuesto_disponible,presupuesto_utilizado,costo_adquisicion,costo_almacenamiento,costo_faltante,costo_total
0,base,central,1.0,1.0,0.95,Optimal,14619,0,14619,0,100.0,0.0,0.0,0.0,194917.940036,0.0,194917.940036
1,demanda_baja,baja,1.0,1.0,0.95,Optimal,11336,0,11336,0,100.0,0.0,0.0,0.0,208604.271944,0.0,208604.271944
2,demanda_alta,alta,1.0,1.0,0.95,Optimal,18407,0,18407,0,100.0,0.0,0.0,0.0,179099.935292,0.0,179099.935292
3,presupuesto_menos_10,central,0.9,1.0,0.95,Optimal,14619,0,14619,0,100.0,0.0,0.0,0.0,194917.940036,0.0,194917.940036
4,presupuesto_mas_10,central,1.1,1.0,0.95,Optimal,14619,0,14619,0,100.0,0.0,0.0,0.0,194917.940036,0.0,194917.940036
5,capacidad_menos_10,central,1.0,0.9,0.95,Optimal,14619,0,14619,0,100.0,0.0,0.0,0.0,194917.940036,0.0,194917.940036
6,capacidad_mas_10,central,1.0,1.1,0.95,Optimal,14619,0,14619,0,100.0,0.0,0.0,0.0,194917.940036,0.0,194917.940036
7,servicio_90,central,1.0,1.0,0.90,Optimal,14619,0,14619,0,100.0,0.0,0.0,0.0,194917.940036,0.0,194917.940036
8,servicio_98,central,1.0,1.0,0.98,Optimal,14619,0,14619,0,100.0,0.0,0.0,0.0,194917.940036,0.0,194917.940036


In [33]:
# ==================================================
# Uso de capacidad por escenario y nodo
# ==================================================

filas_capacidad = []

for resultado in resultados_escenarios:

    escenario_id = resultado["escenario"].iloc[0]
    factor_capacidad = resultado["factor_capacidad"].iloc[0]

    resultado = resultado.copy()

    resultado["inventario_mas_reposicion_equivalente"] = (
        (
            resultado["stock_inicial"]
            + resultado["reposicion_recomendada"]
        )
        * resultado["factor_espacio"]
    )

    resumen_capacidad = (
        resultado
        .groupby("id_tienda", as_index=False)
        .agg(
            capacidad_base=("capacidad_base", "first"),
            inventario_mas_reposicion_equivalente=(
                "inventario_mas_reposicion_equivalente",
                "sum"
            )
        )
    )

    resumen_capacidad["capacidad_disponible_equivalente"] = (
        resumen_capacidad["capacidad_base"]
        * factor_capacidad
    )

    resumen_capacidad["utilizacion_capacidad_pct"] = (
        resumen_capacidad[
            "inventario_mas_reposicion_equivalente"
        ]
        / resumen_capacidad[
            "capacidad_disponible_equivalente"
        ]
        * 100
    )

    resumen_capacidad["restriccion_activa"] = (
        resumen_capacidad[
            "inventario_mas_reposicion_equivalente"
        ]
        >= resumen_capacidad[
            "capacidad_disponible_equivalente"
        ] - 1e-6
    )

    resumen_capacidad["escenario"] = escenario_id

    filas_capacidad.append(
        resumen_capacidad[
            [
                "escenario",
                "id_tienda",
                "capacidad_disponible_equivalente",
                "inventario_mas_reposicion_equivalente",
                "utilizacion_capacidad_pct",
                "restriccion_activa"
            ]
        ]
    )

uso_capacidad = pd.concat(
    filas_capacidad,
    ignore_index=True
)

print("=== USO DE CAPACIDAD ===")
print("Filas:", len(uso_capacidad))
print(
    "Escenarios:",
    uso_capacidad["escenario"].nunique()
)
print(
    "Nodos:",
    uso_capacidad["id_tienda"].nunique()
)
print(
    "Utilización máxima:",
    round(
        uso_capacidad[
            "utilizacion_capacidad_pct"
        ].max(),
        2
    )
)
print(
    "Restricciones activas:",
    uso_capacidad["restriccion_activa"].sum()
)

uso_capacidad.head()

=== USO DE CAPACIDAD ===
Filas: 135
Escenarios: 9
Nodos: 15
Utilización máxima: 80.94
Restricciones activas: 0


,escenario,id_tienda,capacidad_disponible_equivalente,inventario_mas_reposicion_equivalente,utilizacion_capacidad_pct,restriccion_activa
0,base,APP,17396.28,12612.3,72.499983,False
1,base,T001,13401.41,9040.3,67.457827,False
2,base,T002,13578.29,9099.7,67.016539,False
3,base,T003,13781.90,9412.1,68.293196,False
4,base,T004,13609.20,9288.6,68.252359,False


In [34]:
# ==================================================
# Recomendaciones de reposición del escenario base
# ==================================================

recomendaciones_reposicion = resultado_base.copy()

# Nivel de servicio por tienda
nivel_servicio_tienda = (
    recomendaciones_reposicion
    .groupby("id_tienda", as_index=False)
    .agg(
        demanda_total_tienda=("demanda_escenario", "sum"),
        demanda_atendida_tienda=("demanda_atendida", "sum")
    )
)

nivel_servicio_tienda["nivel_servicio_pct"] = (
    nivel_servicio_tienda["demanda_atendida_tienda"]
    / nivel_servicio_tienda["demanda_total_tienda"]
    * 100
)

recomendaciones_reposicion = (
    recomendaciones_reposicion
    .merge(
        nivel_servicio_tienda[
            ["id_tienda", "nivel_servicio_pct"]
        ],
        on="id_tienda",
        how="left",
        validate="many_to_one"
    )
)

# Seleccionar y ordenar las columnas oficiales
recomendaciones_reposicion = recomendaciones_reposicion[
    [
        "escenario",
        "periodo_objetivo",
        "id_tienda",
        "categoria",
        "demanda_escenario",
        "stock_inicial",
        "reposicion_recomendada",
        "demanda_atendida",
        "demanda_no_atendida",
        "stock_final_estimado",
        "nivel_servicio_pct",
        "costo_adquisicion_total",
        "costo_almacenamiento_total",
        "costo_faltante_total",
        "costo_total",
        "estado_solver"
    ]
].copy()

# Renombrar periodo para la salida final
recomendaciones_reposicion = (
    recomendaciones_reposicion.rename(
        columns={"periodo_objetivo": "periodo"}
    )
)

print("=== RECOMENDACIONES DE REPOSICIÓN ===")
print("Filas:", len(recomendaciones_reposicion))
print(
    "Duplicados tienda-categoría:",
    recomendaciones_reposicion.duplicated(
        ["id_tienda", "categoria"]
    ).sum()
)
print(
    "Reposición total:",
    recomendaciones_reposicion[
        "reposicion_recomendada"
    ].sum()
)
print(
    "Faltante total:",
    recomendaciones_reposicion[
        "demanda_no_atendida"
    ].sum()
)
print(
    "Estados del solver:",
    recomendaciones_reposicion[
        "estado_solver"
    ].unique()
)

recomendaciones_reposicion.head()

=== RECOMENDACIONES DE REPOSICIÓN ===
Filas: 90
Duplicados tienda-categoría: 0
Reposición total: 0
Faltante total: 0
Estados del solver: ['Optimal']


,escenario,periodo,id_tienda,categoria,demanda_escenario,stock_inicial,reposicion_recomendada,demanda_atendida,demanda_no_atendida,stock_final_estimado,nivel_servicio_pct,costo_adquisicion_total,costo_almacenamiento_total,costo_faltante_total,costo_total,estado_solver
0,base,2026-01,APP,Abarrotes,919,1712,0,919,0,793,100.0,0.0,539.750439,0.0,539.750439,Optimal
1,base,2026-01,APP,Bebidas,706,1337,0,706,0,631,100.0,0.0,322.153517,0.0,322.153517,Optimal
2,base,2026-01,APP,Cuidado Personal,470,936,0,470,0,466,100.0,0.0,657.897946,0.0,657.897946,Optimal
3,base,2026-01,APP,Electrohogar,439,892,0,439,0,453,100.0,0.0,7726.239176,0.0,7726.239176,Optimal
4,base,2026-01,APP,Hogar,362,659,0,362,0,297,100.0,0.0,1105.345241,0.0,1105.345241,Optimal


In [35]:
# ==================================================
# Exportación de los tres CSV oficiales
# ==================================================

ruta_recomendaciones = (
    RUTA_RESULTADOS / "recomendaciones_reposicion.csv"
)

ruta_resumen = (
    RUTA_RESULTADOS / "resumen_escenarios_optimizacion.csv"
)

ruta_capacidad = (
    RUTA_RESULTADOS / "uso_capacidad_optimizacion.csv"
)

recomendaciones_reposicion.to_csv(
    ruta_recomendaciones,
    index=False,
    encoding="utf-8-sig"
)

resumen_escenarios.to_csv(
    ruta_resumen,
    index=False,
    encoding="utf-8-sig"
)

uso_capacidad.to_csv(
    ruta_capacidad,
    index=False,
    encoding="utf-8-sig"
)

print("=== EXPORTACIÓN COMPLETADA ===")
print("1.", ruta_recomendaciones)
print("2.", ruta_resumen)
print("3.", ruta_capacidad)

=== EXPORTACIÓN COMPLETADA ===
1. c:\Users\DEYSI\OneDrive - Universidad Nacional Mayor de San Marcos\Documentos\proyecto-andinaretail\resultados\recomendaciones_reposicion.csv
2. c:\Users\DEYSI\OneDrive - Universidad Nacional Mayor de San Marcos\Documentos\proyecto-andinaretail\resultados\resumen_escenarios_optimizacion.csv
3. c:\Users\DEYSI\OneDrive - Universidad Nacional Mayor de San Marcos\Documentos\proyecto-andinaretail\resultados\uso_capacidad_optimizacion.csv


In [36]:
# ==================================================
# Validación de archivos exportados
# ==================================================

from pathlib import Path

archivos = [
    "recomendaciones_reposicion.csv",
    "resumen_escenarios_optimizacion.csv",
    "uso_capacidad_optimizacion.csv"
]

print("=== VALIDACIÓN DE ARCHIVOS ===")

for archivo in archivos:

    ruta = RUTA_RESULTADOS / archivo

    existe = ruta.exists()

    if existe:
        df = pd.read_csv(ruta)

        print(f"\n{archivo}")
        print("Existe:", existe)
        print("Filas:", len(df))
        print("Columnas:", len(df.columns))

    else:
        print(f"\n{archivo}")
        print("NO EXISTE")

=== VALIDACIÓN DE ARCHIVOS ===

recomendaciones_reposicion.csv
Existe: True
Filas: 90
Columnas: 16

resumen_escenarios_optimizacion.csv
Existe: True
Filas: 9
Columnas: 17

uso_capacidad_optimizacion.csv
Existe: True
Filas: 135
Columnas: 6
